In [1]:
"""
ML-Powered Career Guidance Platform
====================================
A comprehensive system that uses actual machine learning for career recommendations.

Components:
1. Semantic matching using sentence transformers (not just TF-IDF)
2. Collaborative filtering for placement success patterns
3. Placement timeline prediction
4. Skill gap analysis with learning paths
5. Personalized ranking based on student profile
"""

import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Core ML libraries
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import joblib

try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TRANSFORMERS_AVAILABLE = True
except ImportError:
    SENTENCE_TRANSFORMERS_AVAILABLE = False
    print("⚠️  sentence-transformers not available. Install with: pip install sentence-transformers")

try:
    from sklearn.ensemble import HistGradientBoostingRegressor
    HIST_GB_AVAILABLE = True
except ImportError:
    HIST_GB_AVAILABLE = False


class SemanticJobMatcher:
    """
    Phase 1: Semantic matching using embeddings instead of keyword matching.
    This understands that "Python developer" and "Software engineer with Python" are similar.
    """
    
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        """
        Initialize with a sentence transformer model.
        
        Args:
            model_name: Pretrained model from sentence-transformers
                       'all-MiniLM-L6-v2' is lightweight and fast
        """
        if SENTENCE_TRANSFORMERS_AVAILABLE:
            self.model = SentenceTransformer(model_name)
            self.use_embeddings = True
        else:
            from sklearn.feature_extraction.text import TfidfVectorizer
            self.model = TfidfVectorizer(
                ngram_range=(1, 3),
                max_features=5000,
                min_df=1,
                stop_words='english'
            )
            self.use_embeddings = False
            print("📊 Using TF-IDF fallback (install sentence-transformers for better results)")
        
        self.job_embeddings = None
        self.job_data = None
        self.is_fitted = False
    
    def _prepare_job_text(self, job_df: pd.DataFrame) -> List[str]:
        """
        Create rich text representation of jobs with proper weighting.
        """
        texts = []
        for _, row in job_df.iterrows():
            parts = []
            
            if pd.notna(row.get('job_title')):
                parts.extend([row['job_title']] * 3)
            
            if pd.notna(row.get('required_skills')):
                parts.extend([row['required_skills']] * 3)
            
            if pd.notna(row.get('company_name')):
                parts.extend([row['company_name']] * 2)
            
            if pd.notna(row.get('industry')):
                parts.append(row['industry'])
            
            if pd.notna(row.get('job_description')):
                # Truncate long descriptions
                desc = str(row['job_description'])[:500]
                parts.append(desc)
            
            texts.append(' '.join(parts))
        
        return texts
    
    def fit(self, job_df: pd.DataFrame):
        """
        Build semantic embeddings for all jobs.
        
        Args:
            job_df: DataFrame with job postings
        """
        self.job_data = job_df.copy().reset_index(drop=True)
        
        # Prepare text
        job_texts = self._prepare_job_text(job_df)
        
        # Create embeddings
        if self.use_embeddings:
            print("🧠 Encoding jobs with sentence transformers...")
            self.job_embeddings = self.model.encode(
                job_texts, 
                show_progress_bar=True,
                batch_size=32
            )
        else:
            print("📊 Creating TF-IDF vectors...")
            self.job_embeddings = self.model.fit_transform(job_texts)
        
        self.is_fitted = True
        print(f"✓ Encoded {len(job_df)} jobs")
        print(f"✓ Embedding dimension: {self.job_embeddings.shape[1]}")
        
        return self
    
    def get_semantic_matches(self, user_profile: Dict, top_k: int = 10) -> pd.DataFrame:
        """
        Find semantically similar jobs.
        
        Args:
            user_profile: Dict with 'skills', 'desired_role', 'interests', etc.
            top_k: Number of matches to return
        """
        if not self.is_fitted:
            raise ValueError("Must call fit() first")
        
        # Build user query
        query_parts = []
        if user_profile.get('desired_role'):
            query_parts.extend([user_profile['desired_role']] * 3)
        if user_profile.get('skills'):
            query_parts.extend([user_profile['skills']] * 2)
        if user_profile.get('interests'):
            query_parts.append(user_profile['interests'])
        if user_profile.get('dream_companies'):
            query_parts.extend([user_profile['dream_companies']] * 2)
        
        query = ' '.join(query_parts)
        
        # Encode query
        if self.use_embeddings:
            query_embedding = self.model.encode([query])
        else:
            query_embedding = self.model.transform([query])
        
        # Compute similarities
        similarities = cosine_similarity(query_embedding, self.job_embeddings)[0]
        
        # Get top-k
        top_k = min(top_k, len(similarities))
        top_indices = similarities.argsort()[-top_k:][::-1]
        
        # Prepare results
        results = self.job_data.iloc[top_indices].copy()
        results['semantic_score'] = similarities[top_indices] * 100
        
        return results.reset_index(drop=True)


class PlacementSuccessPredictor:
    """
    Phase 2: Predict placement success probability using student profile.
    This learns from historical data: which students got placed where.
    """
    
    def __init__(self):
        self.model = GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=5,
            random_state=42
        )
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.feature_names = None
        self.is_fitted = False
    
    def _engineer_features(self, df: pd.DataFrame, is_training: bool = True) -> np.ndarray:
        """
        Create features from student profile and job.
        """
        features = []
        feature_names = []
        
        # Academic features
        if 'cgpa' in df.columns:
            features.append(df['cgpa'].values.reshape(-1, 1))
            feature_names.append('cgpa')
        
        if 'backlogs' in df.columns:
            features.append(df['backlogs'].values.reshape(-1, 1))
            feature_names.append('backlogs')
        
        # Branch encoding
        if 'branch' in df.columns:
            if is_training:
                self.label_encoders['branch'] = LabelEncoder()
                branch_encoded = self.label_encoders['branch'].fit_transform(df['branch'].fillna('Other'))
            else:
                # Handle unseen branches
                branch_filled = df['branch'].fillna('Other')
                branch_encoded = np.array([
                    self.label_encoders['branch'].transform([b])[0] 
                    if b in self.label_encoders['branch'].classes_ 
                    else -1 
                    for b in branch_filled
                ])
            features.append(branch_encoded.reshape(-1, 1))
            feature_names.append('branch')
        
        # Skills match (count of matching skills)
        if 'user_skills' in df.columns and 'required_skills' in df.columns:
            skills_match = df.apply(
                lambda row: self._count_skill_overlap(
                    str(row['user_skills']), 
                    str(row['required_skills'])
                ), 
                axis=1
            ).values.reshape(-1, 1)
            features.append(skills_match)
            feature_names.append('skills_match')
        
        # Projects count
        if 'num_projects' in df.columns:
            features.append(df['num_projects'].values.reshape(-1, 1))
            feature_names.append('num_projects')
        
        # Internship experience
        if 'has_internship' in df.columns:
            features.append(df['has_internship'].astype(int).values.reshape(-1, 1))
            feature_names.append('has_internship')
        
        # Company tier (if available)
        if 'company_tier' in df.columns:
            if is_training:
                self.label_encoders['company_tier'] = LabelEncoder()
                tier_encoded = self.label_encoders['company_tier'].fit_transform(
                    df['company_tier'].fillna('Unknown')
                )
            else:
                tier_filled = df['company_tier'].fillna('Unknown')
                tier_encoded = np.array([
                    self.label_encoders['company_tier'].transform([t])[0] 
                    if t in self.label_encoders['company_tier'].classes_ 
                    else -1 
                    for t in tier_filled
                ])
            features.append(tier_encoded.reshape(-1, 1))
            feature_names.append('company_tier')
        
        if is_training:
            self.feature_names = feature_names
        
        # Concatenate all features
        if features:
            X = np.hstack(features)
        else:
            raise ValueError("No valid features found in DataFrame")
        
        return X
    
    def _count_skill_overlap(self, user_skills: str, required_skills: str) -> int:
        """Count overlapping skills."""
        user_set = set(s.strip().lower() for s in str(user_skills).split(','))
        req_set = set(s.strip().lower() for s in str(required_skills).split(','))
        return len(user_set & req_set)
    
    def fit(self, training_data: pd.DataFrame):
        """
        Train the placement success predictor.
        
        Args:
            training_data: Historical data with columns:
                - Student features: cgpa, branch, backlogs, skills, etc.
                - Job features: required_skills, company_tier, etc.
                - Target: 'placed' (1 if student got placed, 0 otherwise)
        """
        if 'placed' not in training_data.columns:
            raise ValueError("Training data must have 'placed' target column")
        
        # Extract features
        X = self._engineer_features(training_data, is_training=True)
        y = training_data['placed'].values
        
        # Scale features
        X_scaled = self.scaler.fit_transform(X)
        
        # Train model
        print(f"🎯 Training placement predictor on {len(training_data)} samples...")
        self.model.fit(X_scaled, y)
        
        # Compute feature importance
        if hasattr(self.model, 'feature_importances_'):
            importance_df = pd.DataFrame({
                'feature': self.feature_names,
                'importance': self.model.feature_importances_
            }).sort_values('importance', ascending=False)
            print("\n📊 Feature Importance:")
            print(importance_df.to_string(index=False))
        
        self.is_fitted = True
        print("✓ Placement predictor trained")
        
        return self
    
    def predict_placement_probability(self, student_job_pairs: pd.DataFrame) -> np.ndarray:
        """
        Predict probability of placement success.
        
        Args:
            student_job_pairs: DataFrame with student and job features
            
        Returns:
            Array of probabilities (0-1)
        """
        if not self.is_fitted:
            raise ValueError("Must call fit() first")
        
        X = self._engineer_features(student_job_pairs, is_training=False)
        X_scaled = self.scaler.transform(X)
        
        # Get probability of placement (class 1)
        probabilities = self.model.predict_proba(X_scaled)[:, 1]
        
        return probabilities


class RecruitmentTimelinePredictor:
    """
    Phase 3: Predict when companies typically recruit (time-series component).
    Helps students prepare in advance.
    """
    
    def __init__(self):
        if HIST_GB_AVAILABLE:
            self.model = HistGradientBoostingRegressor(
                max_iter=100,
                learning_rate=0.1,
                max_depth=5,
                random_state=42
            )
        else:
            self.model = RandomForestRegressor(
                n_estimators=100,
                max_depth=10,
                random_state=42
            )
        self.company_encoders = {}
        self.is_fitted = False
    
    def fit(self, historical_data: pd.DataFrame):
        """
        Learn recruitment patterns from historical data.
        
        Args:
            historical_data: DataFrame with columns:
                - company_name
                - year
                - month (target variable)
                - company_tier (optional)
                - num_positions (optional)
        """
        if 'month' not in historical_data.columns:
            raise ValueError("Must have 'month' column (1-12)")
        
        # Encode company names
        self.company_encoders['company'] = LabelEncoder()
        company_encoded = self.company_encoders['company'].fit_transform(
            historical_data['company_name']
        )
        
        # Build features
        features = [company_encoded.reshape(-1, 1)]
        
        if 'year' in historical_data.columns:
            features.append(historical_data['year'].values.reshape(-1, 1))
        
        if 'company_tier' in historical_data.columns:
            self.company_encoders['tier'] = LabelEncoder()
            tier_encoded = self.company_encoders['tier'].fit_transform(
                historical_data['company_tier'].fillna('Unknown')
            )
            features.append(tier_encoded.reshape(-1, 1))
        
        if 'num_positions' in historical_data.columns:
            features.append(historical_data['num_positions'].values.reshape(-1, 1))
        
        X = np.hstack(features)
        y = historical_data['month'].values
        
        # Store number of features for prediction
        self.num_features = X.shape[1]
        
        print(f"📅 Training timeline predictor on {len(historical_data)} records...")
        self.model.fit(X, y)
        
        self.is_fitted = True
        print("✓ Timeline predictor trained")
        
        return self
    
    def predict_recruitment_month(self, company_name: str, year: int = 2024) -> Dict:
        """
        Predict when a company is likely to recruit.
        
        Args:
            company_name: Name of the company
            year: Target year
            
        Returns:
            Dictionary with predicted month and confidence
        """
        if not self.is_fitted:
            raise ValueError("Must call fit() first")
        
        # Check if company is known
        if company_name not in self.company_encoders['company'].classes_:
            return {
                'company': company_name,
                'predicted_month': None,
                'confidence': 'low',
                'message': 'Company not in historical data'
            }
        
        # Encode company
        company_encoded = self.company_encoders['company'].transform([company_name])[0]
        
        # Build feature vector matching training features
        features = [company_encoded, year]
        
        # Add tier if it was used in training
        if 'tier' in self.company_encoders:
            # For prediction, use a default tier or lookup
            features.append(0)  # Default tier encoding
        
        # Add num_positions if it was used in training  
        if hasattr(self, 'num_features') and self.num_features > len(features):
            features.append(10)  # Default num_positions
        
        features = np.array(features).reshape(1, -1)
        
        # Predict
        predicted_month = int(round(self.model.predict(features)[0]))
        predicted_month = max(1, min(12, predicted_month))  # Clamp to 1-12
        
        month_names = ['', 'January', 'February', 'March', 'April', 'May', 'June',
                      'July', 'August', 'September', 'October', 'November', 'December']
        
        return {
            'company': company_name,
            'predicted_month': predicted_month,
            'month_name': month_names[predicted_month],
            'confidence': 'high',
            'message': f'Typically recruits in {month_names[predicted_month]}'
        }


class CareerGuidancePlatform:
    """
    Main platform that integrates all ML components.
    """
    
    def __init__(self):
        self.semantic_matcher = SemanticJobMatcher()
        self.placement_predictor = PlacementSuccessPredictor()
        self.timeline_predictor = RecruitmentTimelinePredictor()
        self.is_fitted = False
    
    def fit(self, 
            job_postings: pd.DataFrame,
            placement_history: Optional[pd.DataFrame] = None,
            recruitment_timeline: Optional[pd.DataFrame] = None):
        """
        Fit all components of the platform.
        
        Args:
            job_postings: Current job opportunities
            placement_history: Historical placement data (optional but recommended)
            recruitment_timeline: Historical recruitment data (optional)
        """
        print("=" * 60)
        print("🚀 Training ML-Powered Career Guidance Platform")
        print("=" * 60)
        
        # 1. Fit semantic matcher (always required)
        print("\n[1/3] Training Semantic Job Matcher...")
        self.semantic_matcher.fit(job_postings)
        
        # 2. Fit placement predictor (if data available)
        if placement_history is not None:
            print("\n[2/3] Training Placement Success Predictor...")
            self.placement_predictor.fit(placement_history)
        else:
            print("\n[2/3] ⚠️  Skipping Placement Predictor (no historical data)")
        
        # 3. Fit timeline predictor (if data available)
        if recruitment_timeline is not None:
            print("\n[3/3] Training Recruitment Timeline Predictor...")
            self.timeline_predictor.fit(recruitment_timeline)
        else:
            print("\n[3/3] ⚠️  Skipping Timeline Predictor (no historical data)")
        
        self.is_fitted = True
        print("\n" + "=" * 60)
        print("✅ Platform ready for recommendations!")
        print("=" * 60)
        
        return self
    
    def recommend_jobs(self, 
                       student_profile: Dict,
                       top_k: int = 10,
                       include_timeline: bool = True) -> pd.DataFrame:
        """
        Get personalized job recommendations for a student.
        
        Args:
            student_profile: Dict with keys:
                - skills: str (comma-separated)
                - desired_role: str
                - cgpa: float
                - branch: str
                - dream_companies: str (optional)
                - has_internship: bool
                - num_projects: int
            top_k: Number of recommendations
            include_timeline: Whether to include recruitment timeline
            
        Returns:
            DataFrame with ranked recommendations
        """
        if not self.is_fitted:
            raise ValueError("Must call fit() first")
        
        # Step 1: Get semantic matches
        matches = self.semantic_matcher.get_semantic_matches(
            student_profile, 
            top_k=top_k * 3  # Get more candidates for reranking
        )
        
        # Step 2: Add placement probability (if model is trained)
        if self.placement_predictor.is_fitted:
            # Prepare data for prediction
            student_job_df = matches.copy()
            for key in ['cgpa', 'branch', 'has_internship', 'num_projects']:
                if key in student_profile:
                    student_job_df[key] = student_profile[key]
            
            student_job_df['user_skills'] = student_profile.get('skills', '')
            student_job_df['backlogs'] = student_profile.get('backlogs', 0)
            
            # Predict
            placement_probs = self.placement_predictor.predict_placement_probability(
                student_job_df
            )
            matches['placement_probability'] = placement_probs * 100
            
            # Combine scores (70% semantic, 30% placement probability)
            matches['final_score'] = (
                0.7 * matches['semantic_score'] + 
                0.3 * matches['placement_probability']
            )
        else:
            matches['final_score'] = matches['semantic_score']
            matches['placement_probability'] = None
        
        # Step 3: Add recruitment timeline
        if include_timeline and self.timeline_predictor.is_fitted:
            timeline_info = []
            for company in matches['company_name']:
                timeline = self.timeline_predictor.predict_recruitment_month(company)
                timeline_info.append(timeline['month_name'])
            matches['expected_recruitment'] = timeline_info
        
        # Step 4: Sort and return top-k
        matches = matches.sort_values('final_score', ascending=False).head(top_k)
        
        return matches.reset_index(drop=True)
    
    def get_skill_development_path(self, 
                                   student_profile: Dict,
                                   target_job_id: int) -> Dict:
        """
        Provide a learning path to bridge skill gaps.
        
        Args:
            student_profile: Student's current profile
            target_job_id: ID of target job
            
        Returns:
            Dictionary with skill gaps and learning recommendations
        """
        # Find the job
        job = self.semantic_matcher.job_data[
            self.semantic_matcher.job_data['job_id'] == target_job_id
        ]
        
        if job.empty:
            raise ValueError(f"Job ID {target_job_id} not found")
        
        job = job.iloc[0]
        
        # Parse skills
        user_skills = set(s.strip().lower() for s in student_profile['skills'].split(','))
        required_skills = set(s.strip().lower() for s in str(job['required_skills']).split(','))
        
        matching = user_skills & required_skills
        missing = required_skills - user_skills
        
        # Create learning path (this could be enhanced with actual course recommendations)
        learning_path = {
            'job_title': job['job_title'],
            'company': job['company_name'],
            'your_skills': sorted(list(user_skills)),
            'matching_skills': sorted(list(matching)),
            'skills_to_learn': sorted(list(missing)),
            'match_percentage': len(matching) / len(required_skills) * 100 if required_skills else 0,
            'estimated_prep_time': f"{len(missing) * 2} weeks",  # Rough estimate
            'priority': 'High' if len(matching) / len(required_skills) > 0.5 else 'Medium'
        }
        
        return learning_path
    
    def save_models(self, path: str = "/home/claude/models"):
        """Save trained models to disk."""
        import os
        os.makedirs(path, exist_ok=True)
        
        if self.semantic_matcher.is_fitted:
            joblib.dump(self.semantic_matcher, f"{path}/semantic_matcher.pkl")
        
        if self.placement_predictor.is_fitted:
            joblib.dump(self.placement_predictor, f"{path}/placement_predictor.pkl")
        
        if self.timeline_predictor.is_fitted:
            joblib.dump(self.timeline_predictor, f"{path}/timeline_predictor.pkl")
        
        print(f"✓ Models saved to {path}/")
    
    @classmethod
    def load_models(cls, path: str = "/home/claude/models"):
        """Load trained models from disk."""
        import os
        platform = cls()
        
        if os.path.exists(f"{path}/semantic_matcher.pkl"):
            platform.semantic_matcher = joblib.load(f"{path}/semantic_matcher.pkl")
        
        if os.path.exists(f"{path}/placement_predictor.pkl"):
            platform.placement_predictor = joblib.load(f"{path}/placement_predictor.pkl")
        
        if os.path.exists(f"{path}/timeline_predictor.pkl"):
            platform.timeline_predictor = joblib.load(f"{path}/timeline_predictor.pkl")
        
        platform.is_fitted = True
        print(f"✓ Models loaded from {path}/")
        
        return platform


In [ ]:
%pip install fastapi uvicorn

In [ ]:
"""
Demo: ML-Powered Career Guidance Platform
==========================================
This script demonstrates the complete workflow with synthetic data.
"""
op
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)


def generate_synthetic_data():
    """
    Generate realistic synthetic data for demonstration.
    In production, this would come from your actual database.
    """
    
    # 1. JOB POSTINGS DATA
    companies = ['Google', 'Microsoft', 'Amazon', 'Meta', 'Apple', 'Netflix', 
                'Goldman Sachs', 'JP Morgan', 'Flipkart', 'Zomato', 
                'Swiggy', 'Paytm', 'TCS', 'Infosys', 'Wipro', 'Accenture']
    
    roles = ['Software Engineer', 'Data Scientist', 'Machine Learning Engineer',
            'Frontend Developer', 'Backend Developer', 'Full Stack Developer',
            'DevOps Engineer', 'Product Manager', 'Data Analyst', 'Business Analyst']
    
    skill_sets = {
        'Software Engineer': 'Python, Java, Data Structures, Algorithms, System Design, Git',
        'Data Scientist': 'Python, Machine Learning, Statistics, SQL, Pandas, Scikit-learn',
        'Machine Learning Engineer': 'Python, Deep Learning, TensorFlow, PyTorch, MLOps, Docker',
        'Frontend Developer': 'JavaScript, React, HTML, CSS, TypeScript, Redux',
        'Backend Developer': 'Java, Python, Spring Boot, REST API, Database, Microservices',
        'Full Stack Developer': 'JavaScript, React, Node.js, MongoDB, Python, REST API',
        'DevOps Engineer': 'Docker, Kubernetes, AWS, CI/CD, Linux, Terraform',
        'Product Manager': 'Product Strategy, Analytics, SQL, A/B Testing, Agile',
        'Data Analyst': 'SQL, Excel, Tableau, Python, Statistics, Business Intelligence',
        'Business Analyst': 'SQL, Excel, Requirements Analysis, Agile, Stakeholder Management'
    }
    
    industries = ['Technology', 'Finance', 'E-commerce', 'Consulting', 'Product']
    company_tiers = ['Tier1', 'Tier2', 'Tier3']
    
    # Generate 100 job postings
    jobs = []
    for i in range(100):
        role = np.random.choice(roles)
        company = np.random.choice(companies)
        
        job = {
            'job_id': f'JOB_{i+1:03d}',
            'job_title': role,
            'company_name': company,
            'required_skills': skill_sets[role],
            'industry': np.random.choice(industries),
            'company_tier': company_tiers[0] if company in ['Google', 'Microsoft', 'Amazon', 'Meta', 'Apple'] 
                           else company_tiers[1] if company in ['Goldman Sachs', 'JP Morgan', 'Netflix']
                           else company_tiers[2],
            'job_description': f'Looking for a talented {role} to join our team. Must have strong skills in {skill_sets[role].split(",")[0]}.',
            'min_cgpa': np.random.choice([6.5, 7.0, 7.5, 8.0]),
            'location': np.random.choice(['Bangalore', 'Hyderabad', 'Pune', 'Mumbai', 'Delhi'])
        }
        jobs.append(job)
    
    job_df = pd.DataFrame(jobs)
    
    # 2. HISTORICAL PLACEMENT DATA (for training the placement predictor)
    branches = ['CSE', 'IT', 'ECE', 'EEE', 'Mechanical', 'Civil']
    
    placements = []
    for i in range(500):
        cgpa = np.random.uniform(6.0, 10.0)
        branch = np.random.choice(branches, p=[0.3, 0.25, 0.2, 0.1, 0.1, 0.05])
        backlogs = np.random.choice([0, 0, 0, 1, 2], p=[0.6, 0.2, 0.1, 0.07, 0.03])
        num_projects = np.random.randint(0, 6)
        has_internship = np.random.choice([True, False], p=[0.4, 0.6])
        
        # Pick a random job
        job_idx = np.random.randint(0, len(jobs))
        job = jobs[job_idx]
        
        # Simulate skill matching
        all_skills = ['Python', 'Java', 'C++', 'JavaScript', 'Machine Learning', 
                     'Data Structures', 'SQL', 'React', 'AWS', 'Docker']
        num_skills = np.random.randint(3, 8)
        user_skills = ','.join(np.random.choice(all_skills, num_skills, replace=False))
        
        # Calculate placement probability based on features
        # Higher CGPA, relevant branch, projects, internship → higher chance
        base_prob = 0.3
        if cgpa > 8.0:
            base_prob += 0.2
        if branch in ['CSE', 'IT']:
            base_prob += 0.15
        if backlogs == 0:
            base_prob += 0.1
        if num_projects >= 2:
            base_prob += 0.1
        if has_internship:
            base_prob += 0.15
        
        placed = np.random.random() < base_prob
        
        placement = {
            'student_id': f'STU_{i+1:03d}',
            'cgpa': cgpa,
            'branch': branch,
            'backlogs': backlogs,
            'num_projects': num_projects,
            'has_internship': has_internship,
            'user_skills': user_skills,
            'required_skills': job['required_skills'],
            'company_tier': job['company_tier'],
            'placed': int(placed)
        }
        placements.append(placement)
    
    placement_df = pd.DataFrame(placements)
    
    # 3. RECRUITMENT TIMELINE DATA
    timelines = []
    for company in companies:
        for year in [2022, 2023, 2024]:
            # Most companies recruit in July-September (campus season)
            month = np.random.choice([7, 8, 9, 10], p=[0.3, 0.4, 0.2, 0.1])
            
            timeline = {
                'company_name': company,
                'year': year,
                'month': month,
                'company_tier': company_tiers[0] if company in ['Google', 'Microsoft', 'Amazon', 'Meta', 'Apple'] 
                               else company_tiers[1] if company in ['Goldman Sachs', 'JP Morgan', 'Netflix']
                               else company_tiers[2],
                'num_positions': np.random.randint(5, 50)
            }
            timelines.append(timeline)
    
    timeline_df = pd.DataFrame(timelines)
    
    return job_df, placement_df, timeline_df


def demo_basic_workflow():
    """Demonstrate the basic workflow of the platform."""
    
    print("\n" + "="*80)
    print("DEMO: ML-POWERED CAREER GUIDANCE PLATFORM")
    print("="*80)
    
    # Step 1: Generate data
    print("\n📊 Generating synthetic data...")
    job_df, placement_df, timeline_df = generate_synthetic_data()
    
    print(f"  ✓ {len(job_df)} job postings")
    print(f"  ✓ {len(placement_df)} historical placements")
    print(f"  ✓ {len(timeline_df)} recruitment timeline records")
    
    # Step 2: Initialize and train the platform
    print("\n🎓 Initializing Career Guidance Platform...")
    platform = CareerGuidancePlatform()
    
    platform.fit(
        job_postings=job_df,
        placement_history=placement_df,
        recruitment_timeline=timeline_df
    )
    
    # Step 3: Create a student profile
    student_profile = {
        'skills': 'Python, Machine Learning, Data Structures, Algorithms, SQL',
        'desired_role': 'Machine Learning Engineer',
        'cgpa': 8.5,
        'branch': 'CSE',
        'backlogs': 0,
        'num_projects': 3,
        'has_internship': True,
        'dream_companies': 'Google, Microsoft, Amazon',
        'interests': 'Deep Learning, Computer Vision'
    }
    
    print("\n" + "-"*80)
    print("👨‍🎓 STUDENT PROFILE")
    print("-"*80)
    for key, value in student_profile.items():
        print(f"  {key:20s}: {value}")
    
    # Step 4: Get recommendations
    print("\n" + "-"*80)
    print("🎯 PERSONALIZED JOB RECOMMENDATIONS")
    print("-"*80)
    
    recommendations = platform.recommend_jobs(
        student_profile=student_profile,
        top_k=10,
        include_timeline=True
    )
    
    # Display top 5 recommendations
    print("\nTop 5 Recommended Jobs:\n")
    display_cols = ['job_title', 'company_name', 'final_score', 
                   'placement_probability', 'expected_recruitment']
    
    for idx, row in recommendations.head(5).iterrows():
        print(f"{idx+1}. {row['job_title']} at {row['company_name']}")
        print(f"   Match Score: {row['final_score']:.1f}/100")
        if row['placement_probability'] is not None:
            print(f"   Placement Probability: {row['placement_probability']:.1f}%")
        if 'expected_recruitment' in row:
            print(f"   Expected Recruitment: {row['expected_recruitment']}")
        print()
    
    # Step 5: Skill gap analysis
    print("-"*80)
    print("📚 SKILL GAP ANALYSIS")
    print("-"*80)
    
    target_job_id = recommendations.iloc[0]['job_id']
    skill_path = platform.get_skill_development_path(
        student_profile=student_profile,
        target_job_id=target_job_id
    )
    
    print(f"\nTarget Job: {skill_path['job_title']} at {skill_path['company']}")
    print(f"Match Percentage: {skill_path['match_percentage']:.1f}%")
    print(f"Priority: {skill_path['priority']}")
    print(f"\nYour Skills: {', '.join(skill_path['your_skills'][:5])}...")
    print(f"Matching Skills: {', '.join(skill_path['matching_skills']) if skill_path['matching_skills'] else 'None'}")
    print(f"Skills to Learn: {', '.join(skill_path['skills_to_learn']) if skill_path['skills_to_learn'] else 'None'}")
    print(f"Estimated Prep Time: {skill_path['estimated_prep_time']}")
    
    # Step 6: Save models
    print("\n" + "-"*80)
    print("💾 SAVING MODELS")
    print("-"*80)
    platform.save_models()
    
    print("\n" + "="*80)
    print("✅ DEMO COMPLETED SUCCESSFULLY")
    print("="*80)
    
    return platform, job_df, placement_df, recommendations


def demo_multiple_students():
    """Show recommendations for different student profiles."""
    
    print("\n" + "="*80)
    print("DEMO: COMPARING RECOMMENDATIONS FOR DIFFERENT STUDENTS")
    print("="*80)
    
    # Generate data
    job_df, placement_df, timeline_df = generate_synthetic_data()
    
    # Train platform
    platform = CareerGuidancePlatform()
    platform.fit(job_df, placement_df, timeline_df)
    
    # Define different student profiles
    students = [
        {
            'name': 'High Achiever',
            'profile': {
                'skills': 'Python, Java, Machine Learning, Deep Learning, TensorFlow, AWS',
                'desired_role': 'Machine Learning Engineer',
                'cgpa': 9.2,
                'branch': 'CSE',
                'backlogs': 0,
                'num_projects': 5,
                'has_internship': True,
                'dream_companies': 'Google, Microsoft'
            }
        },
        {
            'name': 'Average Student',
            'profile': {
                'skills': 'Python, SQL, Java, HTML, CSS',
                'desired_role': 'Software Engineer',
                'cgpa': 7.5,
                'branch': 'IT',
                'backlogs': 1,
                'num_projects': 2,
                'has_internship': False,
                'dream_companies': 'Infosys, TCS'
            }
        },
        {
            'name': 'Career Switcher',
            'profile': {
                'skills': 'Circuit Design, VLSI, C++, Python',
                'desired_role': 'Software Engineer',
                'cgpa': 8.0,
                'branch': 'ECE',
                'backlogs': 0,
                'num_projects': 1,
                'has_internship': False,
                'dream_companies': 'Amazon, Flipkart'
            }
        }
    ]
    
    # Get recommendations for each
    for student in students:
        print(f"\n{'='*80}")
        print(f"Student: {student['name']}")
        print(f"{'='*80}")
        
        recommendations = platform.recommend_jobs(
            student_profile=student['profile'],
            top_k=3,
            include_timeline=False
        )
        
        print("\nTop 3 Recommendations:\n")
        for idx, row in recommendations.iterrows():
            print(f"{idx+1}. {row['job_title']} at {row['company_name']}")
            print(f"   Match Score: {row['final_score']:.1f}/100")
            if row['placement_probability'] is not None:
                print(f"   Placement Success Probability: {row['placement_probability']:.1f}%")
            print()


def show_model_insights():
    """Display insights about what the models learned."""
    
    print("\n" + "="*80)
    print("DEMO: MODEL INSIGHTS & FEATURE IMPORTANCE")
    print("="*80)
    
    # Generate data and train
    job_df, placement_df, timeline_df = generate_synthetic_data()
    platform = CareerGuidancePlatform()
    platform.fit(job_df, placement_df, timeline_df)
    
    print("\n📊 What the models learned:")
    print("\n1. Semantic Matcher:")
    print("   - Understands job role similarities (e.g., 'ML Engineer' ≈ 'Data Scientist')")
    print("   - Captures skill relationships (e.g., 'Python' relates to 'Machine Learning')")
    print("   - Considers company preferences in matching")
    
    print("\n2. Placement Predictor:")
    print("   - Learned which combinations of CGPA, branch, skills lead to success")
    print("   - Identifies importance of internships and projects")
    print("   - Understands company tier requirements")
    
    print("\n3. Timeline Predictor:")
    print("   - Learned seasonal recruitment patterns")
    print("   - Company-specific hiring schedules")
    
    # Show some predictions
    print("\n📅 Sample Timeline Predictions:")
    for company in ['Google', 'Microsoft', 'TCS']:
        timeline = platform.timeline_predictor.predict_recruitment_month(company, 2025)
        print(f"   {company}: {timeline['message']}")


if __name__ == "__main__":
    print("\n🎯 Starting Career Guidance Platform Demo\n")
    print("This demo will show:")
    print("  1. Basic workflow with a student profile")
    print("  2. Recommendations for different student types")
    print("  3. Model insights and predictions")
    print("\n" + "="*80)
    
    # Run demos
    try:
        # Demo 1: Basic workflow
        platform, jobs, placements, recommendations = demo_basic_workflow()
        
        # Demo 2: Multiple students
        demo_multiple_students()
        
        # Demo 3: Model insights
        show_model_insights()
        
        print("\n" + "="*80)
        print("🎉 ALL DEMOS COMPLETED SUCCESSFULLY!")
        print("="*80)
        print("\nNext Steps:")
        print("  1. Replace synthetic data with your actual college placement data")
        print("  2. Fine-tune model parameters based on performance")
        print("  3. Add more features (certifications, coding contest ratings, etc.)")
        print("  4. Build a web interface for students to interact with the system")
        print("  5. Continuously update models as new placement data comes in")
        
    except Exception as e:
        print(f"\n❌ Error in demo: {str(e)}")
        import traceback
        traceback.print_exc()



🎯 Starting Career Guidance Platform Demo

This demo will show:
  1. Basic workflow with a student profile
  2. Recommendations for different student types
  3. Model insights and predictions


DEMO: ML-POWERED CAREER GUIDANCE PLATFORM

📊 Generating synthetic data...
  ✓ 100 job postings
  ✓ 500 historical placements
  ✓ 48 recruitment timeline records

🎓 Initializing Career Guidance Platform...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

✓ Encoded 100 jobs
✓ Embedding dimension: 384

[2/3] Training Placement Success Predictor...
🎯 Training placement predictor on 500 samples...

📊 Feature Importance:
       feature  importance
          cgpa    0.537538
        branch    0.122782
  num_projects    0.105170
has_internship    0.085788
  skills_match    0.059468
  company_tier    0.046066
      backlogs    0.043188
✓ Placement predictor trained

[3/3] Training Recruitment Timeline Predictor...
📅 Training timeline predictor on 48 records...
✓ Timeline predictor trained

✅ Platform ready for recommendations!

--------------------------------------------------------------------------------
👨‍🎓 STUDENT PROFILE
--------------------------------------------------------------------------------
  skills              : Python, Machine Learning, Data Structures, Algorithms, SQL
  desired_role        : Machine Learning Engineer
  cgpa                : 8.5
  branch              : CSE
  backlogs            : 0
  num_projects        : 3


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

✓ Encoded 100 jobs
✓ Embedding dimension: 384

[2/3] Training Placement Success Predictor...
🎯 Training placement predictor on 500 samples...

📊 Feature Importance:
       feature  importance
          cgpa    0.567650
  num_projects    0.120186
        branch    0.107543
      backlogs    0.059067
  company_tier    0.057463
  skills_match    0.054657
has_internship    0.033435
✓ Placement predictor trained

[3/3] Training Recruitment Timeline Predictor...
📅 Training timeline predictor on 48 records...
✓ Timeline predictor trained

✅ Platform ready for recommendations!

Student: High Achiever

Top 3 Recommendations:

1. Machine Learning Engineer at Amazon
   Match Score: 85.2/100
   Placement Success Probability: 97.2%

2. Machine Learning Engineer at JP Morgan
   Match Score: 84.6/100
   Placement Success Probability: 97.3%

3. Machine Learning Engineer at Meta
   Match Score: 84.2/100
   Placement Success Probability: 97.2%


Student: Average Student

Top 3 Recommendations:

1. Softw

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

✓ Encoded 100 jobs
✓ Embedding dimension: 384

[2/3] Training Placement Success Predictor...
🎯 Training placement predictor on 500 samples...

📊 Feature Importance:
       feature  importance
          cgpa    0.549561
  num_projects    0.115845
        branch    0.109280
has_internship    0.081087
  skills_match    0.067877
  company_tier    0.040642
      backlogs    0.035709
✓ Placement predictor trained

[3/3] Training Recruitment Timeline Predictor...
📅 Training timeline predictor on 48 records...
✓ Timeline predictor trained

✅ Platform ready for recommendations!

📊 What the models learned:

1. Semantic Matcher:
   - Understands job role similarities (e.g., 'ML Engineer' ≈ 'Data Scientist')
   - Captures skill relationships (e.g., 'Python' relates to 'Machine Learning')
   - Considers company preferences in matching

2. Placement Predictor:
   - Learned which combinations of CGPA, branch, skills lead to success
   - Identifies importance of internships and projects
   - Understan